# 01 — Audit dữ liệu nhập vào trước khi train PDP

**Ngày tạo:** 07/10/2026. **Mục đích:** kiểm tra bộ dữ liệu trong `data/archive` trước khi tái dựng PDP và thay backbone bằng ConvNeXt-V2-Base. Notebook này chỉ đọc dữ liệu; không sửa ảnh/annotation, không tải model, không thuê hoặc khởi động GPU.

**Máy train mục tiêu:** RTX 4090 trên **Vast.ai**. Notebook audit có thể chạy trên máy phát triển hoặc Vast.ai; môi trường thật được ghi ở ô đầu. Đây chưa phải notebook training.

**Phạm vi audit:** inventory ảnh; tất cả COCO JSON ở root, `tasks_pdp`, `tasks`, `synth`; checksum annotation; ảnh tham chiếu có tồn tại; ID/label/bbox/area; overlap đường dẫn train/test; so sánh giao thức task. Ảnh được kiểm tra bằng Pillow verify() trên mẫu cố định; chưa giải mã toàn bộ pixel hoặc kiểm tra trùng nội dung bằng hash/perceptual hash cho toàn bộ ảnh.

**Cách chạy:** chạy tuần tự các ô từ trên xuống. Có thể đặt biến `AUTOCHECKOUT_DATA_ROOT` để trỏ đến bản dataset trên Vast.ai. Ô đầu ghi run ID và thời gian bắt đầu; ô cuối ghi thời gian kết thúc, lỗi và việc còn thiếu. Output được lưu cùng notebook. Mọi notebook audit/preprocess/model-check/train/eval tiếp theo cũng phải có mục đích, config, timestamp bắt đầu/kết thúc, kết quả và giới hạn xác minh.


**Cập nhật sau lần chạy này:** người dùng đã chốt protocol **5 task 100+4×25**. Split/task JSON ảnh thật đã nghiệm thu trong [notebook 02](02_split_and_acceptance.ipynb); xem [báo cáo release](../../docs/data_preprocessing/reports/split-100-4x25-2026-10-07.md). Notebook này giữ output của lần audit/phân tích trước quyết định; nhãn synth được review riêng trong [notebook 03](03_annotation_review.ipynb).


In [1]:
import os, sys, json, hashlib, math, platform, subprocess, uuid
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter, defaultdict

TZ = ZoneInfo('Asia/Bangkok')
def timestamp():
    return datetime.now(TZ).isoformat(timespec='seconds')
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pdp/main.py').exists())
DATA_ROOT = Path(os.environ.get('AUTOCHECKOUT_DATA_ROOT', str(REPO / 'data/archive'))).resolve()
assert DATA_ROOT.is_dir(), f'Data root không tồn tại: {DATA_ROOT}'
SAMPLE_PER_SOURCE = 8
run = {
    'run_id': str(uuid.uuid4()), 'purpose': 'read-only imported RPC data audit',
    'started_at': timestamp(), 'finished_at': None, 'timezone': 'Asia/Bangkok',
    'data_root': str(DATA_ROOT), 'python': sys.version.split()[0],
    'platform': platform.platform(), 'host': platform.node(),
    'git_commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip(),
    'git_dirty': bool(subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()),
    'execution_method': 'sequential Python code cells; no training',
    'sample_per_source': SAMPLE_PER_SOURCE,
}
print(json.dumps(run, ensure_ascii=False, indent=2))


{
  "run_id": "1bfc5c33-6a4b-4145-9bc5-24d390362b6f",
  "purpose": "read-only imported RPC data audit",
  "started_at": "2026-10-07T10:46:26+07:00",
  "finished_at": null,
  "timezone": "Asia/Bangkok",
  "data_root": "/home/sakana/Code/AutoCheckout-CL/data/archive",
  "python": "3.14.4",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "host": "Laptop",
  "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
  "git_dirty": true,
  "execution_method": "sequential Python code cells; no training",
  "sample_per_source": 8
}


## 1. Inventory và đường dẫn

Bỏ qua metadata macOS và virtualenv có trong archive. Không thực thi code/scripts đi kèm dataset. Số lượng ảnh là số file thực tế, chưa chứng minh mọi file decode được.


In [2]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
image_files = set()
source_images = defaultdict(list)
inventory = Counter()
for base, dirs, names in os.walk(DATA_ROOT):
    dirs[:] = sorted(d for d in dirs if d not in {'.venv', '__MACOSX', '.git', '__pycache__'})
    for name in sorted(names):
        if name.startswith('.'):
            continue
        path = Path(base) / name
        rel = path.relative_to(DATA_ROOT).as_posix()
        if path.suffix.lower() in IMAGE_EXTENSIONS:
            image_files.add(rel)
            source = '/'.join(Path(rel).parts[:3]) if rel.startswith('synth/') else Path(rel).parts[0]
            inventory[source] += 1
            source_images[source].append(rel)
print('Ảnh theo nguồn:', json.dumps(dict(sorted(inventory.items())), ensure_ascii=False, indent=2))
print('train2019 directory exists:', (DATA_ROOT / 'train2019').is_dir())
print('Ignored imported metadata/environment: .DS_Store, __MACOSX, .venv')


Ảnh theo nguồn: {
  "backgrounds": 209,
  "cutouts": 7513,
  "scripts": 1,
  "synth/group/task1": 5000,
  "synth/group/task2": 5000,
  "synth/group/task3": 5000,
  "synth/group/task4": 5000,
  "test2019": 24000,
  "val2019": 6000
}
train2019 directory exists: False
Ignored imported metadata/environment: .DS_Store, __MACOSX, .venv


## 2. Giao thức task hiện có

So sánh task của nguồn nhập với mapping trong repo. Chưa chọn 4 task hay 5 task cho lần train mới. Nếu dùng 5 task 100+4×25, cần tạo lại task annotations và split sau audit; không đổi `n_tasks` đơn thuần trên JSON 4 task.


In [3]:
protocols = {}
for path in sorted(DATA_ROOT.glob('tasks_pdp/*/rpc_task_info.json')):
    info = json.loads(path.read_text())
    sizes = [t['n_new'] for t in info['tasks']]
    protocols[path.relative_to(DATA_ROOT).as_posix()] = sizes
repo_config = json.loads((REPO / 'configs/tasks_100-4x25_seed0.json').read_text())
repo_sizes = [len(t['classes']) for t in repo_config['tasks'] if not t.get('reserved', False)]
print('Imported:', json.dumps(protocols, indent=2))
print('Repo data tasks:', repo_sizes, '| slots:', repo_config['num_slots'], '| classifier outputs:', repo_config['num_classes'])
print('Protocol mismatch:', any(v != repo_sizes for v in protocols.values()))


Imported: {
  "tasks_pdp/group/rpc_task_info.json": [
    48,
    68,
    47,
    37
  ],
  "tasks_pdp/group_mini/rpc_task_info.json": [
    48,
    68,
    47,
    37
  ],
  "tasks_pdp/group_real/rpc_task_info.json": [
    48,
    68,
    47,
    37
  ]
}
Repo data tasks: [100, 25, 25, 25, 25] | slots: 224 | classifier outputs: 225
Protocol mismatch: True


## 3. COCO integrity, ảnh thiếu và checksum annotation

Đường dẫn ảnh được thử theo root archive, thư mục annotation và nguồn val/test/train tương ứng. Kiểm tra annotation đầy đủ trên các JSON được chọn. Việc crop/clip bbox để sửa lỗi chưa được thực hiện. Chênh lệch `area` với bbox không tự là lỗi vì `area` có thể là diện tích mask.


In [4]:
def resolve_image(filename, ann_path):
    if not isinstance(filename, str):
        return None
    candidates = [DATA_ROOT / filename, ann_path.parent / filename]
    lower = ann_path.name.lower()
    if 'single' in lower or lower == 'instances_train2019.json':
        sources = ['train2019']
    elif 'test' in lower:
        sources = ['test2019']
    else:
        sources = ['val2019', 'train2019']
    candidates.extend(DATA_ROOT / source / filename for source in sources)
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate.is_relative_to(DATA_ROOT):
            rel = candidate.relative_to(DATA_ROOT).as_posix()
            if rel in image_files:
                return rel
    return None

ann_paths = sorted(set(DATA_ROOT.glob('instances_*.json')) |
                   set(DATA_ROOT.glob('tasks_pdp/*/*.json')) |
                   set(DATA_ROOT.glob('tasks/*/*.json')) |
                   set(DATA_ROOT.glob('synth/**/annotations.json')))
audits = []
resolved_by_file = {}
raw_categories = {}
for path in ann_paths:
    payload = path.read_bytes()
    data = json.loads(payload)
    if not all(k in data for k in ['images', 'annotations', 'categories']):
        continue
    images, annotations, categories = data['images'], data['annotations'], data['categories']
    image_ids = [im['id'] for im in images]
    annotation_ids = [a['id'] for a in annotations]
    category_ids = [c['id'] for c in categories]
    image_map = {im['id']: im for im in images}
    category_set = set(category_ids)
    errors = Counter()
    errors['duplicate_image_ids'] = len(image_ids) - len(set(image_ids))
    errors['duplicate_annotation_ids'] = len(annotation_ids) - len(set(annotation_ids))
    errors['duplicate_category_ids'] = len(category_ids) - len(category_set)
    missing = []
    resolved = set()
    for image in images:
        rel = resolve_image(image.get('file_name'), path)
        if rel is None:
            missing.append(image.get('file_name'))
        else:
            resolved.add(rel)
        if image.get('width', 0) <= 0 or image.get('height', 0) <= 0:
            errors['invalid_image_dimensions'] += 1
    errors['missing_images'] = len(missing)
    for ann in annotations:
        image = image_map.get(ann.get('image_id'))
        if image is None:
            errors['unknown_image_ids'] += 1
        if ann.get('category_id') not in category_set:
            errors['unknown_category_ids'] += 1
        box = ann.get('bbox')
        if (not isinstance(box, list) or len(box) != 4 or
                not all(isinstance(x, (int, float)) and math.isfinite(x) for x in box)):
            errors['invalid_bbox'] += 1
            continue
        x, y, w, h = box
        if w <= 0 or h <= 0:
            errors['nonpositive_bbox'] += 1
        if image and (x < -0.01 or y < -0.01 or x+w > image['width']+0.01 or y+h > image['height']+0.01):
            errors['bbox_out_of_bounds'] += 1
        if 'area' not in ann or not isinstance(ann['area'], (int, float)) or not math.isfinite(ann['area']) or ann['area'] <= 0:
            errors['missing_or_invalid_area'] += 1
        if 'iscrowd' not in ann:
            errors['missing_iscrowd'] += 1
    rel = path.relative_to(DATA_ROOT).as_posix()
    record = {'file': rel, 'sha256': hashlib.sha256(payload).hexdigest(),
              'images': len(images), 'annotations': len(annotations), 'categories': len(categories),
              'label_min': min(category_ids) if category_ids else None,
              'label_max': max(category_ids) if category_ids else None,
              'issues': {k:v for k,v in errors.items() if v}, 'missing_examples': missing[:3]}
    audits.append(record)
    resolved_by_file[rel] = resolved
    if rel in ['instances_val2019.json', 'instances_test2019.json']:
        raw_categories[rel] = {c['id']: c['name'] for c in categories}
    del data, payload
print('COCO files audited:', len(audits))
for record in audits:
    if record['file'].startswith('instances_') or record['issues']:
        print(json.dumps(record, ensure_ascii=False))
print('Raw val/test category maps equal:', raw_categories.get('instances_val2019.json') == raw_categories.get('instances_test2019.json'))


COCO files audited: 68
{"file": "instances_test2019.json", "sha256": "2a1cb518b202c7e13a74b4ca742aad76f6246cba788288bac6423c7d4a97ba58", "images": 24000, "annotations": 294333, "categories": 200, "label_min": 1, "label_max": 200, "issues": {}, "missing_examples": []}
{"file": "instances_train2019.json", "sha256": "2fe6891a1f33d54104116940bd2b6167d2e20b846c66808ad33e98cc3775125a", "images": 53739, "annotations": 53739, "categories": 200, "label_min": 1, "label_max": 200, "issues": {"missing_images": 53739}, "missing_examples": ["038900004095_camera0-13.jpg", "038900004095_camera0-18.jpg", "038900004095_camera0-11.jpg"]}
{"file": "instances_val2019.json", "sha256": "25afdfed91bc09bff595399e0876a5707708a7061be3fa4121d13385abd1bde7", "images": 6000, "annotations": 73602, "categories": 200, "label_min": 1, "label_max": 200, "issues": {}, "missing_examples": []}
{"file": "tasks/group/task1_train_single.json", "sha256": "3e2d1f1776e652b663da75f5009312f0802acd8b0bab47dd60d7a674d483dfdc", "imag

## 4. Split overlap và kiểm tra file ảnh trên mẫu cố định

Overlap ở đây so sánh đường dẫn vật lý, chưa phát hiện ảnh chụp gần giống hay cùng nội dung đổi tên. Một ảnh train có thể xuất hiện ở nhiều task theo giao thức IOD; không mặc định đây là leakage train/test.


In [5]:
overlap = {}
for group in ['group', 'group_real', 'group_mini']:
    prefix = f'tasks_pdp/{group}/'
    train_refs, test_refs = set(), set()
    for name, refs in resolved_by_file.items():
        if name.startswith(prefix):
            if Path(name).name.startswith('train_'):
                train_refs.update(refs)
            elif Path(name).name.startswith('test_'):
                test_refs.update(refs)
    intersection = train_refs & test_refs
    overlap[group] = {'unique_train_files': len(train_refs), 'unique_test_files': len(test_refs),
                      'exact_path_overlap': len(intersection), 'examples': sorted(intersection)[:3]}
print('Split overlap:', json.dumps(overlap, indent=2))

image_sample = {'method': 'sorted paths, evenly spaced indices, up to 8 per source', 'checked': 0,
                'decode_errors': [], 'dimension_errors': []}
try:
    from PIL import Image
except ImportError:
    image_sample['skipped'] = 'Pillow is not installed; install to run sampled image verification'
else:
    sampled_sizes = defaultdict(list)
    for source, refs in sorted(source_images.items()):
        refs = sorted(refs)
        n = min(SAMPLE_PER_SOURCE, len(refs))
        indices = sorted({round(i * (len(refs)-1) / max(n-1, 1)) for i in range(n)})
        for index in indices:
            rel = refs[index]
            try:
                with Image.open(DATA_ROOT / rel) as img:
                    size = img.size
                    img.verify()
                sampled_sizes[source].append(list(size))
            except Exception as error:
                image_sample['decode_errors'].append({'file': rel, 'error': str(error)})
            image_sample['checked'] += 1
    image_sample['sizes'] = dict(sampled_sizes)
print('Sampled image check:', json.dumps(image_sample, ensure_ascii=False, indent=2))


Split overlap: {
  "group": {
    "unique_train_files": 26000,
    "unique_test_files": 24000,
    "exact_path_overlap": 0,
    "examples": []
  },
  "group_real": {
    "unique_train_files": 6000,
    "unique_test_files": 24000,
    "exact_path_overlap": 0,
    "examples": []
  },
  "group_mini": {
    "unique_train_files": 1947,
    "unique_test_files": 1000,
    "exact_path_overlap": 0,
    "examples": []
  }
}
Sampled image check: {
  "method": "sorted paths, evenly spaced indices, up to 8 per source",
  "checked": 65,
  "decode_errors": [],
  "dimension_errors": [],
  "sizes": {
    "backgrounds": [
      [
        1840,
        1840
      ],
      [
        1840,
        1840
      ],
      [
        1840,
        1840
      ],
      [
        1840,
        1840
      ],
      [
        1840,
        1840
      ],
      [
        1840,
        1840
      ],
      [
        1840,
        1840
      ],
      [
        1440,
        720
      ]
    ],
    "cutouts": [
      [
      

## 5. Kết quả lần chạy và điều kiện trước training

Output cuối lưu checksum từng annotation, thống kê lỗi, phạm vi sampling và thời gian thực tế. Kiểm tra mẫu dùng Pillow verify(), không phải full pixel decoding. Không coi notebook chạy xong là dataset đã đủ điều kiện train. Chọn protocol/split, kiểm tra full ảnh và ảnh gần trùng, đối chiếu metadata ảnh ghép vẫn là công việc cần tiếp tục. Thiếu ảnh sản phẩm đơn ảnh hưởng các pipeline dùng `train2019`, không tự làm mọi đường train ảnh checkout thất bại.


In [6]:
run['finished_at'] = timestamp()
run['duration_seconds'] = round((datetime.fromisoformat(run['finished_at']) - datetime.fromisoformat(run['started_at'])).total_seconds(), 2)
summary = {
    'run': run, 'image_inventory': dict(sorted(inventory.items())), 'task_protocols': protocols,
    'repo_data_tasks': repo_sizes, 'train2019_exists': (DATA_ROOT/'train2019').is_dir(),
    'coco_files_audited': len(audits), 'coco_files_with_issues': sum(bool(a['issues']) for a in audits),
    'split_path_overlap': overlap, 'sampled_image_check': image_sample,
    'training_gate': 'NOT_READY: task protocol and split policy require confirmation; see per-file issues',
    'limitations': ['full image decode not performed', 'near-duplicate/image-content hashes not checked',
                    'synthetic source split provenance not yet validated', 'no training/GPU verification'],
    'annotation_audit': audits,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "run": {
    "run_id": "1bfc5c33-6a4b-4145-9bc5-24d390362b6f",
    "purpose": "read-only imported RPC data audit",
    "started_at": "2026-10-07T10:46:26+07:00",
    "finished_at": "2026-10-07T10:47:54+07:00",
    "timezone": "Asia/Bangkok",
    "data_root": "/home/sakana/Code/AutoCheckout-CL/data/archive",
    "python": "3.14.4",
    "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
    "host": "Laptop",
    "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
    "git_dirty": true,
    "execution_method": "sequential Python code cells; no training",
    "sample_per_source": 8,
    "duration_seconds": 88.0
  },
  "image_inventory": {
    "backgrounds": 209,
    "cutouts": 7513,
    "scripts": 1,
    "synth/group/task1": 5000,
    "synth/group/task2": 5000,
    "synth/group/task3": 5000,
    "synth/group/task4": 5000,
    "test2019": 24000,
    "val2019": 6000
  },
  "task_protocols": {
    "tasks_pdp/group/rpc_task_info.json": [
      48,
      68,
      47,
      3